# ðŸŒŸ Children's Story Generator
## Image Caption â†’ 20-Line Children's Story | End-to-End Deep Learning Pipeline

---
> **Framework:** PyTorch Â· Hugging Face Transformers Â· PEFT (LoRA/QLoRA)  
> **Base Model:** `TinyLlama/TinyLlama-1.1B-Chat-v1.0`  
> **Dataset:** Synthetically generated using **Claude** (Anthropic API)  
> **Pipeline:** Image â†’ Caption Model â†’ Fine-Tuned LLM â†’ Children's Story  
---

## ðŸ“‹ Table of Contents
1. [Setup & Dependencies](#section1)  
2. [Synthetic Dataset Generation with Claude API](#section2)  
3. [Data Preparation for SFT](#section3)  
4. [Model & Tokenizer Loading](#section4)  
5. [LoRA / QLoRA Configuration](#section5)  
6. [Training & Validation](#section6)  
7. [Save & Load the Fine-Tuned Model](#section7)  
8. [Full Pipeline: Image â†’ Caption â†’ Story](#section8)  
9. [Evaluation: ROUGE & Semantic Similarity](#section9)  
10. [Interactive Demo](#section10)  


---
<a id='section1'></a>
## ðŸ”§ Section 1 â€” Setup & Dependencies

Run this cell once. Restart the kernel afterwards if using Colab or Kaggle.


In [2]:
import subprocess, sys

packages = [
    "groq>=0.9.0",               # Dataset generation via Groq
    "torch",
    "transformers>=4.38.0",
    "peft>=0.10.0",
    "datasets",
    "accelerate",
    "bitsandbytes",
    "trl>=0.8.0",
    "evaluate",
    "rouge-score",
    "nltk",
    "sentence-transformers",
    "scikit-learn",
    "huggingface_hub",
    "Pillow",
]

for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("âœ… All packages installed successfully.")


âœ… All packages installed successfully.


In [3]:
import os, json, re, time, math, random, warnings
warnings.filterwarnings("ignore")

import torch
import numpy as np

import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"ðŸ–¥ï¸  Device  : {DEVICE.upper()}")
print(f"ðŸ”¦  PyTorch : {torch.__version__}")

os.makedirs("data", exist_ok=True)
os.makedirs("outputs/story-lora", exist_ok=True)
print("ðŸ“ Directories created: data/  |  outputs/story-lora/")


ðŸ–¥ï¸  Device  : CUDA
ðŸ”¦  PyTorch : 2.10.0+cu128
ðŸ“ Directories created: data/  |  outputs/story-lora/


---
<a id='section2'></a>
## ðŸ“š Section 2 â€” Synthetic Dataset Generation with Groq API

### Why Synthetic Data?

| Problem | Solution |
|---------|----------|
| No public captionâ†’children's story dataset exists | Generate it with Groq LLM |
| Manual writing is too slow | Groq generates 1 000 pairs in ~5 min (ultra-fast!) |
| Hard to control tone and safety | LLM follows our exact story requirements |
| Need consistent 20-line format | Enforced via structured prompt + JSON output |

### Why Groq?
- âš¡ **Fastest inference API** available (LLaMA-3 at 500+ tokens/sec)
- ðŸ†“ **Free tier** with generous daily limits
- ðŸ”Œ **OpenAI-compatible** client

> ðŸ”‘ Get a free Groq API key at [console.groq.com](https://console.groq.com)  
> Set it as `GROQ_API_KEY` in your environment or Colab Secrets.  
> âš ï¸ Never paste your key directly in the notebook â€” always use env vars!


In [ ]:
from groq import Groq

# Set GROQ_API_KEY in your environment or Colab Secrets; do not hardcode it here.
# from google.colab import userdata
# os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")
if not GROQ_API_KEY:
    raise ValueError("Please set GROQ_API_KEY in your environment or Colab Secrets.")

client = Groq(api_key=GROQ_API_KEY)

# llama-3.3-70b-versatile  -> best quality  (recommended)
# llama-3.1-8b-instant     -> fastest / lowest cost
GROQ_MODEL = "llama-3.3-70b-versatile"
print(f"✅ Groq client ready — model: {GROQ_MODEL}")


âœ… Groq client ready â€” model: llama-3.3-70b-versatile


### Step 2a â€” Auto-Generate 500+ Captions with Groq

Instead of writing captions manually, we ask Groq to generate them in batches.
We define 10 themes, each generating 12 captions per batch across multiple batches.
Target: **600 captions** -> **600 training samples**.


In [5]:
# â”€â”€ 10 Themes for Caption Auto-Generation â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
THEMES = [
    "baby animals in nature (forests, meadows, rivers, meadows)",
    "children playing outdoors in different seasons",
    "fantasy creatures (dragons, fairies, unicorns, gnomes)",
    "ocean and underwater scenes with sea creatures",
    "space, stars, and night sky adventures",
    "farm animals and countryside life",
    "magical forests with talking animals",
    "children on adventures (hiking, sailing, exploring caves)",
    "cozy indoor scenes (fireside, rain, reading, baking)",
    "celebrations and festivals (lanterns, kites, parades)",
]

TARGET_CAPTIONS  = 600    # Total captions to generate
CAPTIONS_PER_BATCH = 12   # Groq generates this many per API call
BATCHES_PER_THEME  = max(1, TARGET_CAPTIONS // (len(THEMES) * CAPTIONS_PER_BATCH))

print(f"âœ… Plan: {len(THEMES)} themes x {BATCHES_PER_THEME} batches x {CAPTIONS_PER_BATCH} captions = {len(THEMES)*BATCHES_PER_THEME*CAPTIONS_PER_BATCH} captions")


âœ… Plan: 10 themes x 5 batches x 12 captions = 600 captions


In [6]:
# â”€â”€ Caption Batch Generator â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

CAPTION_SYSTEM = (
    "You are an expert at writing vivid, imaginative image captions "
    "suitable for children's stories. Each caption describes one warm, "
    "peaceful, magical scene with no violence or adult themes."
)

def generate_captions_batch(theme: str, n: int = 12) -> list:
    '''Ask Groq to generate n diverse captions for a given theme.'''
    prompt = (
        f"Generate exactly {n} unique, vivid image captions about: {theme}\n\n"
        f"Rules:\n"
        f"- Each caption is 1 sentence, 10-20 words long\n"
        f"- Each describes a DIFFERENT specific scene\n"
        f"- Warm, peaceful, child-friendly (ages 4-8)\n"
        f"- Include specific details: colours, animals, objects\n"
        f"- No violence, fear, or adult themes\n\n"
        f"Return ONLY a JSON array of {n} caption strings:\n"
        f'["caption 1", "caption 2", ..., "caption {n}"]'
    )
    try:
        resp = client.chat.completions.create(
            model=GROQ_MODEL,
            max_tokens=800,
            temperature=0.9,
            messages=[
                {"role": "system", "content": CAPTION_SYSTEM},
                {"role": "user",   "content": prompt},
            ]
        )
        raw = resp.choices[0].message.content.strip()
        raw = re.sub(r"^```(?:json)?\s*", "", raw, flags=re.MULTILINE)
        raw = re.sub(r"\s*```$",           "", raw, flags=re.MULTILINE)
        caps = json.loads(raw)
        if isinstance(caps, list):
            return [c.strip() for c in caps if isinstance(c, str) and len(c) > 10]
    except Exception as e:
        print(f"  Warning: batch failed ({e})")
    return []


# â”€â”€ Run Caption Generation â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
CAPTIONS_PATH = "data/all_captions.json"

if os.path.exists(CAPTIONS_PATH):
    with open(CAPTIONS_PATH) as f:
        all_captions = json.load(f)
    print(f"Loaded {len(all_captions)} existing captions. Resuming...")
else:
    all_captions = []

existing_set = set(all_captions)
print("\n Running caption generation (target: " + str(TARGET_CAPTIONS) + ")\n")

for theme in THEMES:
    if len(all_captions) >= TARGET_CAPTIONS:
        break
    for b in range(BATCHES_PER_THEME):
        if len(all_captions) >= TARGET_CAPTIONS:
            break
        print(f"  Theme '{theme[:40]}...' | batch {b+1}")
        batch = generate_captions_batch(theme, CAPTIONS_PER_BATCH)
        new   = [c for c in batch if c not in existing_set]
        all_captions.extend(new)
        existing_set.update(new)
        print(f"    +{len(new)} -> total: {len(all_captions)}")
        time.sleep(0.5)

with open(CAPTIONS_PATH, "w") as f:
    json.dump(all_captions, f, indent=2, ensure_ascii=False)

print("\nâœ… " + str(len(all_captions)) + " captions saved to " + CAPTIONS_PATH)


 Running caption generation (target: 600)

  Theme 'baby animals in nature (forests, meadows...' | batch 1
    +12 -> total: 12
  Theme 'baby animals in nature (forests, meadows...' | batch 2
    +12 -> total: 24
  Theme 'baby animals in nature (forests, meadows...' | batch 3
    +11 -> total: 35
  Theme 'baby animals in nature (forests, meadows...' | batch 4
    +10 -> total: 45
  Theme 'baby animals in nature (forests, meadows...' | batch 5
    +10 -> total: 55
  Theme 'children playing outdoors in different s...' | batch 1
    +12 -> total: 67
  Theme 'children playing outdoors in different s...' | batch 2
    +12 -> total: 79
  Theme 'children playing outdoors in different s...' | batch 3
    +12 -> total: 91
  Theme 'children playing outdoors in different s...' | batch 4
    +12 -> total: 103
  Theme 'children playing outdoors in different s...' | batch 5
    +12 -> total: 115
  Theme 'fantasy creatures (dragons, fairies, uni...' | batch 1
    +12 -> total: 127
  Theme 'fantasy c

In [7]:
# â”€â”€ Preview Generated Captions â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import random as _rnd
print("Total captions: " + str(len(all_captions)) + "\n")
print("10 random samples:")
for cap in _rnd.sample(all_captions, min(10, len(all_captions))):
    print(f"  - {cap}")


Total captions: 593

10 random samples:
  - Children skip rocks on calm blue summer lakes
  - Brown rabbits hop in a green meadow
  - A bright rainbow appears in a night sky filled with glittering diamonds
  - Colourful hot air balloons floating in space
  - Colourful fish gather around treasure
  - Gnomes play with green frogs in a garden.
  - Kids sled down snowy white hills
  - Kites with rainbow tails soar above green fields.
  - Little girls have a tea party under a colourful umbrella.
  - Kids wear yellow helmets in a dark mysterious cave


In [11]:
# â”€â”€ Groq / LLaMA-3 Generation Prompts â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

SYSTEM_PROMPT = (
    "You are a professional children's book author. "
    "Given an image caption, you write exactly 20-line children's stories "
    "that are warm, imaginative, safe, and age-appropriate for children aged 4-8. "
    "You never include violence, fear, or adult themes."
)

# Plain text format â€” no JSON, avoids all parsing issues
USER_TEMPLATE = (
    "Write a children's story of EXACTLY 20 lines based on this caption.\n\n"
    "Caption: {caption}\n\n"
    "Requirements:\n"
    "- EXACTLY 20 lines, each a complete meaningful sentence\n"
    "- Simple, warm English for children aged 4-8\n"
    "- Clear structure: beginning (lines 1-5), middle (6-15), ending (16-20)\n"
    "- Magical, imaginative, emotionally gentle tone\n"
    "- NO violence, horror, fear, or adult themes\n"
    "- Stay faithful to the caption\n"
    "- End with joy, comfort, wonder, or belonging\n\n"
    "Output the 20 lines directly, one per line. No numbering. No JSON. No extra text."
)

print("âœ… Prompts defined")

âœ… Prompts defined


In [36]:
# â”€â”€ Story Generation Function â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def generate_story_for_caption(caption: str, max_retries: int = 3):
    '''Call Groq API to generate a 20-line story. Returns str or None.'''
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=GROQ_MODEL,
                max_tokens=1024,
                temperature=0.75,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": USER_TEMPLATE.format(caption=caption)},
                ]
            )
            raw = response.choices[0].message.content.strip()

            # Clean up any numbering like "1." or "1)" the model adds
            lines = []
            for line in raw.split("\n"):
                line = line.strip()
                if not line:
                    continue
                line = re.sub(r"^\d+[\.\)]\s*", "", line)  # remove "1. " prefix
                if line:
                    lines.append(line)

            if len(lines) >= 15:
                return "\n".join(lines[:20])

            print(f"    âš ï¸  Only {len(lines)} lines (attempt {attempt+1}), retryingâ€¦")

        except Exception as e:
            print(f"    âŒ API error (attempt {attempt+1}): {e}")
            time.sleep(2 ** attempt)

    return None


# â”€â”€ Smoke test â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
print("Running smoke testâ€¦")
_test = generate_story_for_caption("A small rabbit sits beside a sunflower in a golden meadow.")
if _test:
    _lines = _test.split("\n")
    print(f"âœ… Test passed! {len(_lines)} lines generated.")
    print(f"   First line : {_lines[0]}")
    print(f"   Last line  : {_lines[-1]}")
else:
    print("âŒ Test failed â€” check your API key and network.")

Running smoke testâ€¦
âœ… Test passed! 20 lines generated.
   First line : In a beautiful meadow, a small rabbit loved to play outside every day.
   Last line  : The sunflower stood tall, watching over the rabbit as she disappeared into the golden light, feeling happy and at peace.


In [37]:
# â”€â”€ Continue Dataset Generation (resumes from where it stopped) â”€â”€â”€
DATASET_PATH = "data/caption_stories.jsonl"
DELAY_SECS   = 0.3

# Load already saved samples
existing = []
if os.path.exists(DATASET_PATH):
    with open(DATASET_PATH, "r", encoding="utf-8") as f:
        existing = [json.loads(l) for l in f if l.strip()]

existing_captions = {s["caption"] for s in existing}
new_samples, failed = [], []
total     = len(all_captions)
remaining = total - len(existing_captions)

print(f"ðŸ’¾ Already saved : {len(existing_captions)}", flush=True)
print(f"ðŸŽ¯ Remaining     : {remaining}", flush=True)
print(f"ðŸ“Š Total target  : {total}\n", flush=True)

dataset_file = open(DATASET_PATH, "a", encoding="utf-8")

try:
    for idx, caption in enumerate(all_captions, 1):

        # Skip already done
        if caption in existing_captions:
            continue

        print(f"  [{idx:03d}/{total}] {caption[:60]}...", flush=True)
        story = generate_story_for_caption(caption)

        if story:
            sample = {"caption": caption, "story": story}
            new_samples.append(sample)
            dataset_file.write(json.dumps(sample, ensure_ascii=False) + "\n")
            dataset_file.flush()
            n = len([l for l in story.split("\n") if l.strip()])
            print(f"           âœ… {n} lines", flush=True)
        else:
            failed.append(caption)
            print(f"           âŒ FAILED â€” skipped", flush=True)

        time.sleep(DELAY_SECS)

        # Progress every 25 captions
        done = len(existing_captions) + len(new_samples)
        if len(new_samples) % 25 == 0 and len(new_samples) > 0:
            print(f"\n  ðŸ“Š Progress: {done}/{total} ({done/total*100:.0f}%) "
                  f"| New: {len(new_samples)} | Failed: {len(failed)}\n", flush=True)

except KeyboardInterrupt:
    print("\nâš ï¸  Stopped â€” progress saved. Re-run to continue.", flush=True)

finally:
    dataset_file.close()

# â”€â”€ Final summary â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
total_saved = len(existing) + len(new_samples)
print("\n" + "="*55, flush=True)
print(f"  âœ… Generated this run : {len(new_samples)}", flush=True)
print(f"  âŒ Failed / skipped   : {len(failed)}", flush=True)
print(f"  ðŸ’¾ Total in dataset   : {total_saved} / {total}", flush=True)
print(f"  ðŸ“ File               : {DATASET_PATH}", flush=True)
if total_saved < total:
    print(f"\n  â–¶ï¸  Re-run this cell to continue ({total - total_saved} left)", flush=True)
else:
    print(f"\n  ðŸŽ‰ Dataset complete! All {total} stories generated.", flush=True)

ðŸ’¾ Already saved : 242
ðŸŽ¯ Remaining     : 351
ðŸ“Š Total target  : 593

  [243/593] Green trees sparkle with firefly lights...
           âœ… 20 lines
  [244/593] Colourful kites soar under a starry sky...
           âœ… 20 lines
  [245/593] Little foxes chase shooting stars together...
           âœ… 20 lines
  [246/593] Sparkling stars reflect in a quiet lake...
           âœ… 20 lines
  [247/593] Golden stars twinkle above a sleeping lion...
           âœ… 20 lines
  [248/593] Purple night sky with sparkling fireflies...
           âœ… 20 lines
  [249/593] Rainbow comet soaring over green hills...
           âœ… 20 lines
  [250/593] Silver moon shining on a calm lake...
           âœ… 20 lines
  [251/593] Colourful hot air balloons floating in space...
           âœ… 20 lines
  [252/593] Happy astronauts playing with fluffy kittens...
           âœ… 20 lines
  [253/593] Glowing constellations guiding a little rabbit...
           âœ… 20 lines
  [254/593] Bright yellow sun rising

In [38]:
# â”€â”€ Load & Verify Full Dataset â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
with open(DATASET_PATH, "r", encoding="utf-8") as f:
    RAW_DATASET = [json.loads(l) for l in f if l.strip()]

counts = [len([l for l in s["story"].split("\n") if l.strip()]) for s in RAW_DATASET]

print(f"ðŸ“Š Dataset Statistics")
print(f"   Total samples      : {len(RAW_DATASET)}")
print(f"   Exactly 20 lines   : {sum(1 for n in counts if n == 20)}")
print(f"   15â€“19 lines        : {sum(1 for n in counts if 15 <= n < 20)}")
print(f"   < 15 lines (review): {sum(1 for n in counts if n < 15)}")
print(f"   Avg lines          : {sum(counts)/len(counts):.1f}")
print()
print("Sample entries:")
for i, s in enumerate(RAW_DATASET[:3], 1):
    n = len([l for l in s["story"].split("\n") if l.strip()])
    print(f"  [{i}] ({n:2d} lines) {s['caption'][:68]}â€¦")


ðŸ“Š Dataset Statistics
   Total samples      : 401
   Exactly 20 lines   : 401
   15â€“19 lines        : 0
   < 15 lines (review): 0
   Avg lines          : 20.0

Sample entries:
  [1] (20 lines) Yellow ducklings swim in a sunny riverâ€¦
  [2] (20 lines) Fluffy white rabbits play in a green meadowâ€¦
  [3] (20 lines) Brown fawns nestle in a forest of tall treesâ€¦


---
<a id='section3'></a>
## ðŸ—‚ï¸ Section 3 â€” Data Preparation for Supervised Fine-Tuning (SFT)

### SFT Prompt Format
The model learns to **complete** this template â€” only the story tokens contribute to loss:

```
### Caption:
{caption}

### Children's Story (exactly 20 lines):
{story}   â† model generates this
```


In [39]:
from datasets import Dataset

# â”€â”€ Prompt Templates â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def make_training_prompt(caption: str, story: str) -> str:
    '''Full training prompt â€” includes the story target.'''
    return (
        f"### Caption:\n{caption}\n\n"
        f"### Children's Story (exactly 20 lines):\n{story}"
    )

def make_inference_prompt(caption: str) -> str:
    '''Inference prompt â€” model must complete the story.'''
    return (
        f"### Caption:\n{caption}\n\n"
        f"### Children's Story (exactly 20 lines):\n"
    )

def prepare_hf_dataset(raw_data: list) -> Dataset:
    '''Wrap caption-story pairs into HuggingFace Dataset for SFTTrainer.'''
    return Dataset.from_list([
        {"text": make_training_prompt(s["caption"], s["story"])}
        for s in raw_data
    ])

# Preview
sample_prompt = make_training_prompt(RAW_DATASET[0]["caption"], RAW_DATASET[0]["story"])
print("Training prompt preview (first 420 chars):")
print("-" * 65)
print(sample_prompt[:420])
print("â€¦")
print(f"\nApprox tokens : ~{len(sample_prompt)//4}")


Training prompt preview (first 420 chars):
-----------------------------------------------------------------
### Caption:
Yellow ducklings swim in a sunny river

### Children's Story (exactly 20 lines):
In a sunny river, where the water sparkled bright, a family of yellow ducklings loved to swim and play.
The little ducklings were so happy to be together, waddling along the riverbank on a warm day.
Their mother duck watched over them with loving eyes, making sure they were safe and sound.
The yellow ducklings quacked with d
â€¦

Approx tokens : ~530


In [40]:
import random
random.seed(42)

shuffled  = RAW_DATASET[:]
random.shuffle(shuffled)

split_idx     = max(1, int(len(shuffled) * 0.85))
TRAIN_DATA    = shuffled[:split_idx]
VAL_DATA      = shuffled[split_idx:]

train_dataset = prepare_hf_dataset(TRAIN_DATA)
val_dataset   = prepare_hf_dataset(VAL_DATA)

print(f"âœ… Dataset split complete")
print(f"   Training samples   : {len(train_dataset)}")
print(f"   Validation samples : {len(val_dataset)}")


âœ… Dataset split complete
   Training samples   : 340
   Validation samples : 61


---
<a id='section4'></a>
## ðŸ¤– Section 4 â€” Model & Tokenizer Loading

### Why TinyLlama-1.1B?

| Criterion | Detail |
|-----------|--------|
| **Size** | 1.1B params â€” fits on a free Colab T4 (16 GB VRAM) |
| **Architecture** | Llama-2 â€” fully LoRA-compatible |
| **Speed** | Fast to fine-tune and infer |
| **Drop-in replacements** | `Llama-3.2-3B-Instruct`, `Mistral-7B-Instruct-v0.2` |


In [41]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME  = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
USE_4BIT    = torch.cuda.is_available()   # QLoRA on GPU, float32 on CPU
MAX_SEQ_LEN = 1024
OUTPUT_DIR  = "outputs/story-lora"

print(f"ðŸ“¦ Base model : {MODEL_NAME}")
print(f"âš™ï¸  4-bit mode : {USE_4BIT}  (QLoRA)")
print(f"ðŸ“ Max seq len: {MAX_SEQ_LEN}")


ðŸ“¦ Base model : TinyLlama/TinyLlama-1.1B-Chat-v1.0
âš™ï¸  4-bit mode : True  (QLoRA)
ðŸ“ Max seq len: 1024


In [42]:
# â”€â”€ Tokenizer â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

tokenizer.padding_side = "right"   # Required for causal LMs

print(f"âœ… Tokenizer loaded")
print(f"   Vocab size : {tokenizer.vocab_size:,}")
print(f"   EOS token  : {tokenizer.eos_token!r} (id={tokenizer.eos_token_id})")
print(f"   PAD token  : {tokenizer.pad_token!r} (id={tokenizer.pad_token_id})")


âœ… Tokenizer loaded
   Vocab size : 32,000
   EOS token  : '</s>' (id=2)
   PAD token  : '</s>' (id=2)


In [43]:
# â”€â”€ Base Model â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    print("âœ… Model loaded in 4-bit QLoRA mode (GPU)")
else:
    bnb_config = None
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float32,
        device_map="cpu",
        trust_remote_code=True,
    )
    print("âœ… Model loaded in float32 CPU mode")

base_model.config.use_cache      = False
base_model.config.pretraining_tp = 1

total = sum(p.numel() for p in base_model.parameters())
print(f"   Total parameters : {total/1e9:.2f}B")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

âœ… Model loaded in 4-bit QLoRA mode (GPU)
   Total parameters : 0.62B


---
<a id='section5'></a>
## âš™ï¸ Section 5 â€” LoRA / QLoRA Configuration

### How LoRA Works

Instead of updating all 1.1B weights, LoRA adds **tiny trainable matrices** to selected layers:

```
W_new = W_frozen  +  B Â· A
  where  B âˆˆ â„^(dÃ—r),  A âˆˆ â„^(rÃ—k),  r â‰ª min(d, k)
```

| Hyperparameter | Value | Meaning |
|----------------|-------|---------|
| `r` | 16 | Rank of adapter matrices |
| `lora_alpha` | 32 | Scaling factor (Î±/r = 2.0) |
| `lora_dropout` | 0.05 | Regularisation |
| `target_modules` | q,k,v,o,gate,up,down | All attention + FFN layers |

Result: only **~2â€“4 M parameters** are trained â€” a **99 % reduction** in compute.


In [44]:
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

if USE_4BIT:
    base_model = prepare_model_for_kbit_training(base_model)
    print("âœ… Model prepared for k-bit training")

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()


âœ… Model prepared for k-bit training
trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338


---
<a id='section6'></a>
## ðŸ‹ï¸ Section 6 â€” Training & Validation

### Strategy
- **`SFTTrainer`** (from `trl`) applies prompt-completion masking automatically  
- Only the story completion tokens contribute to the cross-entropy loss  
- Best model checkpoint is saved based on validation loss  

### Estimated Training Time

| Setup | ~Time |
|-------|-------|
| Colab T4 GPU Â· 60 samples Â· 3 epochs | 3â€“6 min |
| Colab T4 GPU Â· 500 samples Â· 3 epochs | 20â€“35 min |
| CPU only | â‰¥ 1 hr (reduce epochs) |


In [45]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,

    # â”€â”€ Core â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,         # Effective batch = 4
    gradient_checkpointing=True,

    # â”€â”€ Optimiser â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    optim="paged_adamw_8bit" if USE_4BIT else "adamw_torch",
    learning_rate=2e-4,
    weight_decay=0.001,
    max_grad_norm=0.3,
    warmup_steps=10,
    lr_scheduler_type="cosine",

    # â”€â”€ Precision â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    fp16=False,
    bf16=USE_4BIT,

    # â”€â”€ Logging & checkpointing â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    save_steps=50,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",

    # â”€â”€ SFT â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    dataset_text_field="text",
    max_length=MAX_SEQ_LEN,
    packing=False,
    group_by_length=True,
)

trainer = SFTTrainer(
    model=base_model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    peft_config=lora_config,
    processing_class=tokenizer,
    args=training_args,
)

print("âœ… SFTTrainer configured and ready")
print(f"   Train samples : {len(train_dataset)}")
print(f"   Val samples   : {len(val_dataset)}")


Adding EOS to train dataset:   0%|          | 0/340 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/340 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/61 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/61 [00:00<?, ? examples/s]

âœ… SFTTrainer configured and ready
   Train samples : 340
   Val samples   : 61


In [46]:
# â”€â”€ TRAIN â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
print("ðŸš€ Starting trainingâ€¦\n")
train_result = trainer.train()

print("\nâœ… Training complete!")
print(f"   Training loss : {train_result.training_loss:.4f}")
print(f"   Runtime       : {train_result.metrics.get('train_runtime', 0):.1f} s")

trainer.log_metrics("train", train_result.metrics)
trainer.save_metrics("train", train_result.metrics)

# â”€â”€ VALIDATE â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
eval_results = trainer.evaluate()
perplexity   = math.exp(eval_results.get("eval_loss", float("inf")))

print("\nðŸ“Š Validation Results:")
for k, v in eval_results.items():
    print(f"   {k:<32}: {v:.4f}" if isinstance(v, float) else f"   {k}: {v}")

print(f"\n   Perplexity : {perplexity:.2f}")
print("   (< 20 is good for this creative-generation task)")


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


ðŸš€ Starting trainingâ€¦



Step,Training Loss,Validation Loss
50,1.072721,1.019186
100,0.822839,0.956031
150,0.807192,0.926940
200,0.723202,0.918073
250,0.684444,0.916185



âœ… Training complete!
   Training loss : 0.8853
   Runtime       : 1989.8 s
***** train metrics *****
  total_flos               =  2765539GF
  train_loss               =     0.8853
  train_runtime            = 0:33:09.78
  train_samples_per_second =      0.513
  train_steps_per_second   =      0.128



ðŸ“Š Validation Results:
   eval_loss                       : 0.9162
   eval_runtime                    : 34.6555
   eval_samples_per_second         : 1.7600
   eval_steps_per_second           : 1.7600

   Perplexity : 2.50
   (< 20 is good for this creative-generation task)


---
<a id='section7'></a>
## ðŸ’¾ Section 7 â€” Save & Load the Fine-Tuned Model

We save **only the LoRA adapter** (a few MB) â€” not the full 1.1 B model.  
At inference time we reload the frozen base model and attach the adapter on top.


In [47]:
ADAPTER_PATH = os.path.join(OUTPUT_DIR, "lora_adapter")
trainer.model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)

print(f"âœ… LoRA adapter saved to: {ADAPTER_PATH}\n")
for fn in sorted(os.listdir(ADAPTER_PATH)):
    kb = os.path.getsize(os.path.join(ADAPTER_PATH, fn)) / 1024
    print(f"   {fn:<45}  {kb:8.1f} KB")


âœ… LoRA adapter saved to: outputs/story-lora/lora_adapter

   README.md                                           5.1 KB
   adapter_config.json                                 1.1 KB
   adapter_model.safetensors                       49319.9 KB
   chat_template.jinja                                 0.4 KB
   tokenizer.json                                   3534.2 KB
   tokenizer_config.json                               0.4 KB


In [48]:
from peft import PeftModel

def load_finetuned_model(base_name: str, adapter_path: str, use_4bit: bool = False):
    '''Reload base model and attach LoRA adapter for inference.'''
    tok = AutoTokenizer.from_pretrained(adapter_path, trust_remote_code=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    if use_4bit:
        bnb = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
        )
        base = AutoModelForCausalLM.from_pretrained(
            base_name, quantization_config=bnb,
            device_map="auto", trust_remote_code=True
        )
    else:
        base = AutoModelForCausalLM.from_pretrained(
            base_name, torch_dtype=torch.float32,
            device_map="cpu", trust_remote_code=True
        )

    merged = PeftModel.from_pretrained(base, adapter_path)
    merged.eval()
    return merged, tok


inf_model, inf_tokenizer = load_finetuned_model(MODEL_NAME, ADAPTER_PATH, USE_4BIT)
print("âœ… Fine-tuned model loaded and ready for inference")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

âœ… Fine-tuned model loaded and ready for inference


---
<a id='section8'></a>
## ðŸ”— Section 8 â€” Full Pipeline: Image â†’ Caption â†’ Story

### Architecture
```
â”Œâ”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”     â”Œâ”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”     â”Œâ”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”
â”‚  Image File   â”‚â”€â”€â”€â”€â–¶â”‚  Caption Model     â”‚â”€â”€â”€â”€â–¶â”‚  Story LLM (LoRA)      â”‚
â”‚  or URL       â”‚     â”‚  BLIP / Your Model â”‚     â”‚  TinyLlama + Adapter   â”‚
â””â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”˜     â””â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”˜     â””â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”˜
                              â”‚                            â”‚
                       "A rabbit sitsâ€¦"          "Once upon a timeâ€¦"
```

`StoryGeneratorPipeline` supports three usage modes:
1. **Caption â†’ Story** (you provide the caption directly)
2. **Image â†’ Story** (uses BLIP internally for captioning)  
3. **Your caption model â†’ Story** (plug in via `set_caption_model()`)


In [49]:
# â”€â”€ Core story generation function â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def generate_story(
    caption        : str,
    model,
    tokenizer,
    max_new_tokens : int   = 600,
    temperature    : float = 0.75,
    top_p          : float = 0.92,
    rep_penalty    : float = 1.2,
    num_lines      : int   = 20,
) -> str:
    '''Generate a 20-line children's story from a caption string.'''

    prompt = make_inference_prompt(caption)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=256)
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            repetition_penalty=rep_penalty,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    new_ids  = outputs[0][inputs["input_ids"].shape[-1]:]
    raw_text = tokenizer.decode(new_ids, skip_special_tokens=True).strip()

    lines = [ln.strip() for ln in raw_text.split("\n") if ln.strip()]
    lines = lines[:num_lines]

    # Safety pad (rarely needed after fine-tuning)
    while len(lines) < num_lines:
        lines.append("And they lived happily ever after, full of joy and wonder.")

    return "\n".join(lines)

print("âœ… generate_story() ready")


âœ… generate_story() ready


In [50]:
# â”€â”€ StoryGeneratorPipeline class â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

class StoryGeneratorPipeline:
    '''
    End-to-end pipeline: Image (or caption) â†’ Children's Story.

    Quick start:
        pipeline = StoryGeneratorPipeline(inf_model, inf_tokenizer)

        # Mode 1: from a caption string
        result = pipeline.from_caption("A rabbit sits in a meadow.")

        # Mode 2: from an image file or URL (auto-captions with BLIP)
        result = pipeline.from_image("photo.jpg")

        # Mode 3: plug in YOUR caption model
        pipeline.set_caption_model(your_model, your_processor)
        result = pipeline.from_image("photo.jpg")

        StoryGeneratorPipeline.display(result)
    '''

    def __init__(self, story_model, story_tokenizer):
        self.story_model      = story_model
        self.story_tokenizer  = story_tokenizer
        self._cap_model       = None
        self._cap_proc        = None
        self._cap_type        = None

    # â”€â”€ Caption model management â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

    def load_blip(self):
        '''Load the default BLIP captioner (downloaded on first call).'''
        print("ðŸ“· Loading BLIP captionerâ€¦")
        from transformers import BlipProcessor, BlipForConditionalGeneration
        self._cap_proc  = BlipProcessor.from_pretrained(
            "Salesforce/blip-image-captioning-base"
        )
        self._cap_model = BlipForConditionalGeneration.from_pretrained(
            "Salesforce/blip-image-captioning-base"
        ).to(DEVICE)
        self._cap_type  = "blip"
        print("âœ… BLIP captioner loaded")

    def set_caption_model(self, model, processor, model_type: str = "blip"):
        '''
        Plug in YOUR own caption model.

        Args:
            model      : Your captioning model (already loaded & moved to device)
            processor  : Matching processor / tokenizer
            model_type : "blip" (default) or "custom"
        '''
        self._cap_model = model
        self._cap_proc  = processor
        self._cap_type  = model_type
        print(f"âœ… Custom caption model registered (type='{model_type}')")

    # â”€â”€ Captioning â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

    def caption_image(self, image_input) -> str:
        '''Produce a text caption from an image path, URL, or PIL Image.'''
        from PIL import Image
        import requests

        if self._cap_model is None:
            self.load_blip()

        if isinstance(image_input, str):
            if image_input.startswith("http"):
                img = Image.open(
                    requests.get(image_input, stream=True).raw
                ).convert("RGB")
            else:
                img = Image.open(image_input).convert("RGB")
        else:
            img = image_input.convert("RGB")

        inputs = self._cap_proc(images=img, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            out = self._cap_model.generate(**inputs, max_new_tokens=60)
        caption = self._cap_proc.decode(out[0], skip_special_tokens=True)
        return caption.strip()

    # â”€â”€ Story generation â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

    def from_caption(self, caption: str, **kwargs) -> dict:
        '''Generate a story directly from a caption string.'''
        story = generate_story(
            caption, self.story_model, self.story_tokenizer, **kwargs
        )
        return {"caption": caption, "story": story}

    def from_image(self, image_input, **kwargs) -> dict:
        '''Full end-to-end: image â†’ caption â†’ story.'''
        caption = self.caption_image(image_input)
        story   = generate_story(
            caption, self.story_model, self.story_tokenizer, **kwargs
        )
        return {"caption": caption, "story": story}

    # â”€â”€ Display â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

    @staticmethod
    def display(result: dict):
        '''Pretty-print a caption + story result.'''
        print("=" * 72)
        print(f"ðŸ“· Caption : {result['caption']}")
        print("=" * 72)
        print("ðŸ“– Story:\n")
        for i, line in enumerate(result["story"].split("\n"), 1):
            if line.strip():
                print(f"  {i:2d}. {line}")
        print("=" * 72)


pipeline = StoryGeneratorPipeline(inf_model, inf_tokenizer)
print("âœ… StoryGeneratorPipeline ready\n")
print("Usage:")
print('  result = pipeline.from_caption("A rabbit sits in a meadow.")')
print('  result = pipeline.from_image("photo.jpg")')
print('  StoryGeneratorPipeline.display(result)')


âœ… StoryGeneratorPipeline ready

Usage:
  result = pipeline.from_caption("A rabbit sits in a meadow.")
  result = pipeline.from_image("photo.jpg")
  StoryGeneratorPipeline.display(result)


In [51]:
# â”€â”€ Test the pipeline on 3 captions â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

TEST_CAPTIONS = [
    "A small rabbit sits beside a sunflower in a golden meadow.",
    "A little astronaut bounces across the surface of the moon.",
    "A baby penguin stands on an iceberg watching colourful fish swim below.",
]

for cap in TEST_CAPTIONS:
    result = pipeline.from_caption(cap)
    StoryGeneratorPipeline.display(result)
    print()


ðŸ“· Caption : A small rabbit sits beside a sunflower in a golden meadow.
ðŸ“– Story:

   1. In a beautiful garden, surrounded by lovely flowers, a little rabbit loved to play and explore.
   2. The warm sunshine made her feel happy every day.
   3. She would often sit on the soft grass or climb up high trees for a view of the world below.
   4. One bright morning, she found herself sitting near a big old sunflower with its tall petals glowing like gold.
   5. The sunflower was so big that it seemed to be talking to her, asking if she wanted to share its secrets.
   6. As she sat there, she felt the gentle breeze rustling through the petals, making them sway back and forth.
   7. The rabbit felt as though she had become part of this magical scene, connected to nature in a special way.
   8. The sunflower started to sing a lullaby, and the rabbit listened with wide eyes, feeling its magic fill her heart.
   9. The sound of the song was like music to her ears, and she began to move along

In [52]:
# â”€â”€ Plug In YOUR Caption Model â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Uncomment and adapt the block below to connect your own model.
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

# from transformers import BlipProcessor, BlipForConditionalGeneration
#
# YOUR_MODEL_PATH = "path/to/your/caption/model"
#
# your_proc  = BlipProcessor.from_pretrained(YOUR_MODEL_PATH)
# your_model = BlipForConditionalGeneration.from_pretrained(
#     YOUR_MODEL_PATH
# ).to(DEVICE)
#
# pipeline.set_caption_model(your_model, your_proc, model_type="blip")
#
# # Full end-to-end test
# result = pipeline.from_image("your_test_image.jpg")
# StoryGeneratorPipeline.display(result)

print("ðŸ“Œ Uncomment the block above to plug in your own caption model.")
print("   Then call  pipeline.from_image('your_image.jpg')  for end-to-end use.")


ðŸ“Œ Uncomment the block above to plug in your own caption model.
   Then call  pipeline.from_image('your_image.jpg')  for end-to-end use.


---
<a id='section9'></a>
## ðŸ“Š Section 9 â€” Evaluation: ROUGE & Semantic Similarity

| Metric | What it measures | Target |
|--------|-----------------|--------|
| **ROUGE-1** | Unigram overlap with reference | > 0.30 |
| **ROUGE-L** | Longest common subsequence recall | > 0.20 |
| **Semantic Similarity** | Cosine similarity of sentence embeddings | > 0.75 |
| **BLEU-4** | 4-gram precision | > 0.10 |

> âš ï¸ **Semantic Similarity is the most reliable metric for creative text.**  
> BLEU/ROUGE penalise valid paraphrases â€” two equally good stories can score low.


In [53]:
import evaluate as hf_evaluate
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity as cos_sim

rouge_metric = hf_evaluate.load("rouge")

print("Loading sentence embedding model (all-MiniLM-L6-v2)â€¦")
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
print("âœ… All evaluation tools loaded\n")

# â”€â”€ Metric helpers â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def compute_bleu(reference: str, generated: str) -> float:
    '''BLEU-4 with smoothing.'''
    import nltk
    ref = nltk.word_tokenize(reference.lower())
    hyp = nltk.word_tokenize(generated.lower())
    return round(sentence_bleu(
        [ref], hyp,
        weights=(0.25, 0.25, 0.25, 0.25),
        smoothing_function=SmoothingFunction().method1
    ), 4)

def compute_rouge(reference: str, generated: str) -> dict:
    '''ROUGE-1, ROUGE-2, ROUGE-L.'''
    result = rouge_metric.compute(
        predictions=[generated], references=[reference]
    )
    return {k: round(v, 4) for k, v in result.items()}

def compute_semantic_sim(text_a: str, text_b: str) -> float:
    '''Cosine similarity between sentence embeddings.'''
    ea = embed_model.encode([text_a])
    eb = embed_model.encode([text_b])
    return round(float(cos_sim(ea, eb)[0][0]), 4)

print("compute_bleu() | compute_rouge() | compute_semantic_sim() â€” ready")


Loading sentence embedding model (all-MiniLM-L6-v2)â€¦


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

âœ… All evaluation tools loaded

compute_bleu() | compute_rouge() | compute_semantic_sim() â€” ready


In [54]:
# â”€â”€ Evaluate on validation set â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
print(f"{'#':<4} {'BLEU-4':>7} {'ROUGE-1':>8} {'ROUGE-L':>8} {'Sem.Sim':>8}  Caption")
print("-" * 82)

all_b, all_r1, all_rl, all_ss = [], [], [], []

for i, sample in enumerate(VAL_DATA, 1):
    ref = sample["story"]
    gen = generate_story(sample["caption"], inf_model, inf_tokenizer)
    b   = compute_bleu(ref, gen)
    r   = compute_rouge(ref, gen)
    ss  = compute_semantic_sim(ref, gen)

    all_b.append(b); all_r1.append(r["rouge1"])
    all_rl.append(r["rougeL"]); all_ss.append(ss)

    cap = (sample["caption"][:40] + "â€¦") if len(sample["caption"]) > 40 else sample["caption"]
    print(f"{i:<4} {b:>7.4f} {r['rouge1']:>8.4f} {r['rougeL']:>8.4f} {ss:>8.4f}  {cap}")

n = len(VAL_DATA)
print("-" * 82)
print(f"{'AVG':<4} {sum(all_b)/n:>7.4f} {sum(all_r1)/n:>8.4f} "
      f"{sum(all_rl)/n:>8.4f} {sum(all_ss)/n:>8.4f}")
print(
    "\nInterpretation:\n"
    "  BLEU-4  > 0.10  -- acceptable n-gram overlap for creative generation\n"
    "  ROUGE-L > 0.20  -- good content coverage\n"
    "  Sem.Sim > 0.75  -- strong meaning alignment (most reliable signal)\n"
)


#     BLEU-4  ROUGE-1  ROUGE-L  Sem.Sim  Caption
----------------------------------------------------------------------------------
1     0.0934   0.5304   0.2300   0.8462  White chickens roam free outdoors
2     0.1476   0.4907   0.2468   0.7757  Gnomes tend to a beautiful, vibrant flowâ€¦
3     0.0541   0.4361   0.2000   0.7350  Silver moonbeams dance on a purple galaxâ€¦
4     0.0918   0.4527   0.2128   0.8377  Tiny frogs hop on a yellow lily pad
5     0.0857   0.4514   0.2132   0.7940  Yellow submarines explore shipwrecks
6     0.1414   0.5486   0.2392   0.8412  Children ride bicycles on a sunny springâ€¦
7     0.0777   0.4601   0.2053   0.8387  A yellow rocket soars past a bright oranâ€¦
8     0.0598   0.4244   0.1965   0.6979  Silver moon glows on a purple galaxy
9     0.0692   0.4457   0.1857   0.7688  Curious ducks waddle on quiet forest patâ€¦
10    0.0906   0.4954   0.2092   0.6985  Gnomes read books in a cozy little libraâ€¦
11    0.1387   0.4924   0.2519   0.8707  Colourful

KeyboardInterrupt: 

In [56]:
# â”€â”€ Download the LoRA adapter to your computer â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import shutil, os
from google.colab import files

# Zip the adapter folder
adapter_path = "outputs/story-lora/lora_adapter"
zip_path     = "story_lora_adapter.zip"

shutil.make_archive("story_lora_adapter", "zip", adapter_path)

print(f"âœ… Zipped! Size: {os.path.getsize(zip_path)/1024/1024:.1f} MB")
print("â¬‡ï¸  Downloading now...")

# Auto-download to your PC
files.download(zip_path)

âœ… Zipped! Size: 22.9 MB
â¬‡ï¸  Downloading now...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [57]:
# â”€â”€ Zero-Shot vs Fine-Tuned Comparison â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
COMPARE_CAP = "A baby elephant splashes happily in a shallow blue river."
print(f"Caption: {COMPARE_CAP}\n")

# Zero-shot
zs_prompt = f"Write a short children's story about: {COMPARE_CAP}"
zs_in     = inf_tokenizer(zs_prompt, return_tensors="pt")
zs_device = next(inf_model.parameters()).device
zs_in     = {k: v.to(zs_device) for k, v in zs_in.items()}
with torch.no_grad():
    zs_out = inf_model.generate(**zs_in, max_new_tokens=250,
                                do_sample=True, temperature=0.8,
                                eos_token_id=inf_tokenizer.eos_token_id,
                                pad_token_id=inf_tokenizer.pad_token_id)
zs_new   = zs_out[0][zs_in["input_ids"].shape[-1]:]
zs_text  = inf_tokenizer.decode(zs_new, skip_special_tokens=True).strip()
zs_lines = [l.strip() for l in zs_text.split("\n") if l.strip()]

# Fine-tuned
ft_lines = generate_story(COMPARE_CAP, inf_model, inf_tokenizer).split("\n")

print("â”€â”€ ZERO-SHOT (first 5 lines) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€")
for j, l in enumerate(zs_lines[:5], 1): print(f"  {j}. {l}")
print(f"  â€¦ ({len(zs_lines)} total lines)\n")

print("â”€â”€ FINE-TUNED (first 5 lines) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€")
for j, l in enumerate(ft_lines[:5], 1): print(f"  {j}. {l}")
print(f"  â€¦ ({len(ft_lines)} total lines)")


Caption: A baby elephant splashes happily in a shallow blue river.

â”€â”€ ZERO-SHOT (first 5 lines) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
  1. The sun is shining brightly, and the sky is a very gentle color of blue.
  2. The baby elephant is holding a big watering can with a long handle.
  3. The can is filled with fresh, cool water, and the baby elephant is about to splash and splash some more.
  4. The baby elephant is wearing a soft, fluffy pink hat with a bow around its neck.
  5. The baby elephant's mother is watching over them, smiling and happy to see how much they love to play in the blue river.
  â€¦ (10 total lines)

â”€â”€ FINE-TUNED (first 5 lines) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
  1. In a sunny land, there was a beautiful baby elephant with soft brown skin and big eyes that sparkled

---
<a id='section10'></a>
## ðŸŽ‰ Section 10 â€” Interactive Demo

Try any caption and generate your own story!


In [58]:
# â”€â”€ Try Your Own Caption â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Change the caption below and run the cell!

YOUR_CAPTION = "A little fox cub explores a snowy forest for the very first time."

result = pipeline.from_caption(YOUR_CAPTION)
StoryGeneratorPipeline.display(result)


ðŸ“· Caption : A little fox cub explores a snowy forest for the very first time.
ðŸ“– Story:

   1. In a cold winter, a little fox cub was excited to explore the snowy forest for the very first time.
   2. The trees were covered with soft white flakes that glowed like diamonds in the sunlight.
   3. The little fox cub couldn't wait to see what else the forest had to offer.
   4. As she wandered through the forest, the little fox cub felt happy and curious.
   5. She discovered a hidden stream where she could drink from the warm water without getting muddy.
   6. The sound of the stream flowing was soothing, and it made her feel safe and cozy.
   7. She also found a big tree that seemed like a throne, all twisted branches and big leaves.
   8. The little fox cub thought this would be a perfect place to sit down and rest.
   9. But as she sat there, she noticed some interesting things happening around her.
  10. A rabbit hopped by, its little ears twitching with curiosity.
  11. And a fa

In [ ]:
# â”€â”€ Generate from a Real Image URL â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Uncomment to test full end-to-end with a real image.

# IMAGE_URL = "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/1200px-Cat03.jpg"
# result    = pipeline.from_image(IMAGE_URL)
# StoryGeneratorPipeline.display(result)

print("ðŸ“Œ Uncomment the lines above to run the full Image â†’ Story pipeline.")


In [ ]:
# â”€â”€ Save a Story to File â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def save_story(caption: str, story: str, path: str = "my_story.txt"):
    '''Save a caption + story to a nicely formatted text file.'''
    with open(path, "w", encoding="utf-8") as f:
        f.write(f"Caption: {caption}\n")
        f.write("=" * 60 + "\n\n")
        f.write("Story:\n\n")
        for i, line in enumerate(story.split("\n"), 1):
            if line.strip():
                f.write(f"{i:2d}. {line}\n")
    print(f"âœ… Story saved to '{path}'")

save_story(result["caption"], result["story"])


---

## ðŸ Project Complete!

### What You Built

```
Image â”€â”€â–¶ [Your Caption Model] â”€â”€â–¶ Caption â”€â”€â–¶ [Fine-Tuned TinyLlama + LoRA] â”€â”€â–¶ 20-Line Children's Story
```

### Key Achievements

| Step | What Was Done |
|------|--------------|
| **Dataset** | 60 + caption â†’ story pairs generated via Claude API |
| **Fine-tuning** | TinyLlama with LoRA â€” only ~2 M of 1.1 B params trained |
| **Pipeline** | Modular class â€” plug in any caption model with one line |
| **Evaluation** | BLEU-4, ROUGE-L, Semantic Similarity on held-out val set |

### Next Steps to Improve Quality

- Generate **1 000+ samples** â€” just add more seed captions in Section 2  
- Swap to **`Llama-3.2-3B-Instruct`** for higher story quality  
- Add **DPO** (Direct Preference Optimization) with human story ratings  
- Deploy with **Gradio** for a shareable web demo  
- Add **BLIP-2** for richer, more descriptive captions  

---
> *"Every story begins with a single caption â€” and a model that learns to dream."*
